In [1]:
from datasets import load_dataset
import json

ds = load_dataset("LLM-Digital-Twin/Twin-2K-500", "full_persona")

shuffled = ds["data"].shuffle(seed=0)
persona_summaries = shuffled[:1000]["persona_summary"]

with open("personas1000_verbose.json", "w", encoding="utf-8") as f:
    json.dump(persona_summaries, f, ensure_ascii=False, indent=2)

/Users/batuel/Documents/Projects/ppp/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import re

FIELD_MAP = {
    "Geographic region": "geographic_region",
    "Gender": "gender",
    "Age": "age",
    "Education level": "education_level",
    "Race": "race",
    "Citizen of the US": "us_citizen",
    "Marital status": "marital_status",
    "Religion": "religion",
    "Religious attendance": "religious_attendance",
    "Political affiliation": "political_affiliation",
    "Income": "income",
    "Political views": "political_views",
    "Household size": "household_size",
    "Employment status": "employment_status",
}

def extract_persona_fields(text: str) -> dict:
    result = {}
    for label, key in FIELD_MAP.items():
        match = re.search(rf"{re.escape(label)}:\s*(.+)", text)
        if match:
            value = match.group(1).strip()
            # remove parenthetical clarifications for region if desired
            if key == "geographic_region":
                value = value.split(" (")[0]
            result[key] = value
    return result

persona_demographics = [extract_persona_fields(persona_summaries[i]) for i in range(len(persona_summaries))]
assert len(persona_demographics) == 1000

with open("demographics1000.json", "w", encoding="utf-8") as f:
    json.dump(persona_demographics, f, ensure_ascii=False, indent=2)

In [3]:
assert len(persona_demographics[:800]) == 800
assert len(persona_demographics[800:]) == 200

with open("demographics_train.json", "w", encoding="utf-8") as f:
    json.dump(persona_demographics[:800], f, ensure_ascii=False, indent=2)
with open("demographics_test.json", "w", encoding="utf-8") as f:
    json.dump(persona_demographics[800:], f, ensure_ascii=False, indent=2)

In [17]:
import openai
from typing import List, Dict, Tuple
client = openai.OpenAI()

def extract_xml_field(response: str,
                      output_field: str) -> str:
    try: 
        return response.split(f"<{output_field}>")[1].split(f"</{output_field}>")[0]
    except:
        return None

def get_response_from_openai_messages(messages: List[Dict[str,str]], model: str) -> str:
    resp = client.chat.completions.create(
        model=model,
        temperature=0.7,
        messages=messages,
    )
    return resp.choices[0].message.content

In [18]:
PERSONA_COMPRESSION_PROMPT = lambda persona : f"""# Task
You are given a high-dimensional psychological and demographic profile of a person.
Your task is to compress this into a short, usable description that preserves
behaviorally and decision-relevant traits while discarding test names, scores,
and redundant details.

# Persona
{persona}

# Instructions:
Output a single paragraph inside <persona>...</persona> tags. Do not output anything else."""

def make_persona_compression_messages(current_persona: str):
    messages = [{"role" : "system", "content" : "You are a helpful assistant."},
               {"role" : "user" , "content" : PERSONA_COMPRESSION_PROMPT(current_persona)}]
    return messages

def make_persona_compression_pipeline(current_persona: str):
    current_messages = make_persona_compression_messages(current_persona=current_persona)
    response = get_response_from_openai_messages(messages=current_messages, model="gpt-5.2-2025-12-11")
    persona = extract_xml_field(response=response, output_field="persona")
    return persona.strip()

In [19]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

persona_summaries = persona_summaries[:]
personas_compressed = [None] * len(persona_summaries)

with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(make_persona_compression_pipeline, persona_summaries[i]): i
        for i in range(len(persona_summaries))
    }

    for fut in tqdm(as_completed(futures), total=len(futures), desc="Compressing personas"):
        i = futures[fut]
        personas_compressed[i] = fut.result()

assert len(personas_compressed) == 1000

with open("personas1000.json", "w", encoding="utf-8") as f:
    json.dump(personas_compressed, f, ensure_ascii=False, indent=2)


Compressing personas: 100%|██████████| 1000/1000 [03:18<00:00,  5.03it/s]


In [20]:
assert len(personas_compressed[:800]) == 800
assert len(personas_compressed[800:]) == 200

with open("personas_train.json", "w", encoding="utf-8") as f:
    json.dump(personas_compressed[:800], f, ensure_ascii=False, indent=2)
with open("personas_test.json", "w", encoding="utf-8") as f:
    json.dump(personas_compressed[800:], f, ensure_ascii=False, indent=2)